In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np


In [3]:
# Load the ANN Trained model, scaler and encoder pickle files
model = load_model('ann_churn_model.h5')

# Load the scaler and encoder
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('le_gender.pkl', 'rb') as f:
    encoder_gender = pickle.load(f)

with open('ohe_geography.pkl', 'rb') as f:
    encoder_geography = pickle.load(f)



In [9]:
# Example new data for prediction
input_data = {
    'CreditScore': 600,
    'Geography': 'France',          
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

# Convert input data to DataFrame
input_df = pd.DataFrame([input_data])

## Preprocess the input data

# This is for Gender
input_df['Gender'] = encoder_gender.transform(input_df['Gender'])

# This is for Geography
geography_encoded = encoder_geography.transform(input_df[['Geography']])    
geography_df = pd.DataFrame(geography_encoded, columns=encoder_geography.get_feature_names_out(['Geography']))

# Concatenate the encoded geography back to the input dataframe and drop the original 'Geography' column
input_df = pd.concat([input_df.drop('Geography', axis=1), geography_df], axis=1)
input_df

# Scale the input data
input_df_scaled = scaler.transform(input_df)
input_df_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221, -0.57946723,
        -0.57638802]])

In [12]:
# Predict using the trained model
prediction = model.predict(input_df_scaled)
prediction_class = (prediction > 0.5).astype(int)

if prediction_class[0][0] == 1:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")

1/1 [==============================] - 0s 9ms/step
The customer is not likely to churn.
